# Fatigue modeling

Ordinal models for `fatigue_num` (0–5) with participant-level held-out test, GroupKFold CV, and Optuna tuning. Core logic lives in `src/modeling/`. §2 base uses **20 daily features** (15 numeric + 5 categorical from `FEATURE_COLUMNS`, including mcPHASES `daily_glucose`, `daily_hrv`, `sleep_score` from the processed CSV). Tree models one-hot encode `phase` to 23 columns. §3 History appends the **3 selected history features** (`HISTORY_FEATURES` in `config.py`).


In [1]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    HISTORY_FEATURES,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
    TIME_COL,
    TIME_SERIES_GROUP_COLS,
)
from modeling.data import (
    build_split_bundle,
    load_fatigue_data,
    participant_strata,
    preprocess_after_split,
    split_participant_ids,
    split_summary_table,
)
from modeling.registry import ORDINAL_MODELS
from modeling.runner import tune_and_benchmark_model
from modeling.summaries import (
    CATEGORY_ORDER,
    build_history_ablation_summary,
    collect_categorized_summaries,
    collect_summaries,
)


## 1. Load data and split

Participants are held out with a **stratified split** on per-participant **mean fatigue** (`fatigue_num` averaged over each participant's days) so train/val and test have similar average fatigue levels.

“We randomly assign whole participants to train/val or test, but we do it in a way that both groups contain a similar mix of people with low, medium, and high average fatigue — not just a random 8 people who might all happen to be high-average-fatigue reporters.”

**Post-split preprocessing:** NaNs in `menstrual_health_literacy_num` and `daily_hrv` are filled with the **train/val median only** via `preprocess_after_split()` — test participants never contribute to that statistic.


In [3]:
df = load_fatigue_data('../../' + DATA_PATH)
df = df.sort_values(TIME_SERIES_GROUP_COLS + [TIME_COL]).reset_index(drop=True)

strata = participant_strata(df)
train_val_ids, test_ids = split_participant_ids(df['id'].unique(), strata=strata)
train_val_mask = df['id'].isin(train_val_ids)
test_mask = df['id'].isin(test_ids)

literacy_col = 'menstrual_health_literacy_num'
print(f'Literacy NaNs before preprocess: {df[literacy_col].isna().sum()}')
df = preprocess_after_split(df, train_val_mask)
print(f'Literacy NaNs after preprocess: {df[literacy_col].isna().sum()}')

bundle = build_split_bundle(df, train_val_ids, test_ids, train_val_mask, test_mask)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))


Literacy NaNs before preprocess: 80
Literacy NaNs after preprocess: 0
Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue
0,train_val,34,2659,2.462204
1,test,8,672,2.653274


Test participant ids: [np.int64(7), np.int64(14), np.int64(24), np.int64(38), np.int64(40), np.int64(41), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [4]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
history_ordinal_results = []

ordinal_best_params = {}
history_best_params = {}


## 2. Baseline benchmarks

Simple predictors evaluated with the same GroupKFold CV and held-out test protocol as the tuned models. Includes persistence baselines **`lag1_fatigue`** and **`expanding_mean`**.


In [5]:
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results)

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])


Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.406250,1.640721,-0.188402,0.000000
global_mode,1.156250,1.544479,-0.053072,0.000000
lag1_fatigue,0.950893,1.424175,0.104593,0.549449
expanding_mean,1.025298,1.336863,0.211017,0.422289


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

## 3. Train/Tune models

### Ordinal Regression

Continuous loss on `fatigue_num`, then round and clip to [0, 5].

#### `linear_regression`


In [6]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    # feature_set defaults to 'base' (17 daily features only)
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression  test_mae=1.3452


#### `ordinal_rf`


In [7]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf  test_mae=1.3423


#### `catboost_regressor`


In [8]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor  test_mae=1.2158


GEE models (`gee_gaussian`, `gee_ordinal`) are in [`unused models.ipynb`](unused%20models.ipynb) — kept for longitudinal inference benchmarks, excluded from the main prediction comparison.


### Ordinal Classification

Ordered likelihood or threshold structure on `fatigue_num` 0–5. Evaluated with the same MAE / QWK metrics as regression models.

#### `ordered_logistic`


In [9]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic  test_mae=1.3095


#### `ordinal_forest`


In [10]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest  test_mae=1.2247


#### `population_ordered_logistic`


This model does not assume a different baseline for each participant -- this is because we want the model to generalize to the population.

In training, this model only uses day-varying features and deliberately drops participant-level constants such as age, age_of_first_menarche, etc. The reason is that the model does not want to rely on participant-specific demographics.

In [11]:
_name = 'population_ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] population_ordered_logistic  test_mae=1.4554


#### `catboost_ordinal`


In [12]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal  test_mae=1.1310


### History

Same seven ordinal models as above, with the **3-feature history subset** (`HISTORY_FEATURES` from `config.py`) appended to the daily feature matrix. This subset was selected after history construction tuning and forward selection in [`history feature engineering.ipynb`](history%20feature%20engineering.ipynb); the 3-col vs 7-col comparison and significance analysis are archived in [`3.5 history features.ipynb`](3.5%20history%20features.ipynb).

History construction uses `EWMA_ALPHA` and `ROLLING_WINDOWS` from `config.py` via `build_split_bundle`; first-day NaNs in history columns are imputed with the train/val median.

**History features** (3 cols):
- fatigue lag1: Yesterday's fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person


#### Ordinal Regression (history)


##### `linear_regression` (history)


In [13]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_FEATURES,
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression_history  test_mae=0.8839


##### `ordinal_rf` (history)


In [14]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_FEATURES,
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf_history  test_mae=0.8914


##### `catboost_regressor` (history)


In [15]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_FEATURES,
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor_history  test_mae=0.9107


#### Ordinal Classification (history)


##### `ordered_logistic` (history)


In [16]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_FEATURES,
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic_history  test_mae=0.8839


##### `ordinal_forest` (history)


In [17]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_FEATURES,
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest_history  test_mae=0.8914


##### `population_ordered_logistic` (history)


In [18]:
_name = 'population_ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_FEATURES,
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] population_ordered_logistic_history  test_mae=0.8899


##### `catboost_ordinal` (history)


In [19]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_FEATURES,
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal_history  test_mae=0.8884


## 4. Results summary

Aggregates §2 baselines plus any §3 models run (base and `_history` variants). Results are grouped into **baseline**, **base**, and **history** categories.

The next cell prints **CV** tables in order baseline → base → history, then **test** tables in the same order. Within each table, rows are sorted by `cv_mae` or `test_mae` respectively. The following cell compares base vs history test MAE.


In [20]:
# Merge baselines (§2), base tuned models (§3), and history variants (§3 History).
# globals().get(...) allows partial notebook runs without NameError on skipped cells.

ordinal_results = globals().get('ordinal_results', [])
history_ordinal_results = globals().get('history_ordinal_results', [])
ordinal_best_params = globals().get('ordinal_best_params', {})
history_best_params = globals().get('history_best_params', {})

ran_tuned_models = sorted(set(ordinal_best_params) | set(history_best_params))
print(f'Ran {len(ran_tuned_models)} tuned ordinal models: {ran_tuned_models}')

all_ordinal_results = (
    ordinal_baseline_results
    + ordinal_results
    + history_ordinal_results
)

ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results)
category_summaries = collect_categorized_summaries(all_ordinal_results)

print('CV (sorted by cv_mae within each category; cv_* = mean over GroupKFold folds on train/val)')
for category in CATEGORY_ORDER:
    cv_cat, _ = category_summaries[category]
    if cv_cat.empty:
        continue
    print(f'  {category}')
    display(cv_cat)

print('Test (sorted by test_mae within each category; refit on full train/val, scored on test participants)')
for category in CATEGORY_ORDER:
    _, test_cat = category_summaries[category]
    if test_cat.empty:
        continue
    print(f'  {category}')
    display(test_cat)


Ran 14 tuned ordinal models: ['catboost_ordinal', 'catboost_ordinal_history', 'catboost_regressor', 'catboost_regressor_history', 'linear_regression', 'linear_regression_history', 'ordered_logistic', 'ordered_logistic_history', 'ordinal_forest', 'ordinal_forest_history', 'ordinal_rf', 'ordinal_rf_history', 'population_ordered_logistic', 'population_ordered_logistic_history']
CV (sorted by cv_mae within each category; cv_* = mean over GroupKFold folds on train/val)
  baseline


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
lag1_fatigue,{},0.824015,1.315205,0.096947,0.546287,0.193110
expanding_mean,{},0.867974,1.198179,0.280799,0.495990,0.127344
global_mode,{},1.216046,1.554387,-0.200378,0.000000,0.243084
global_mean,{},1.348516,1.606173,-0.297646,0.000000,0.153930


  base


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
catboost_ordinal,"{'iterations': 120, 'depth': 6, 'learning_rate...",1.175757,1.428995,-0.024966,0.150758,0.074122
catboost_regressor,"{'iterations': 135, 'depth': 5, 'learning_rate...",1.230680,1.496874,-0.134853,0.112228,0.144843
ordinal_forest,"{'n_estimators': 231, 'max_depth': 14, 'min_sa...",1.232167,1.502062,-0.126583,0.114155,0.133816
ordinal_rf,"{'n_estimators': 107, 'max_depth': 5, 'min_sam...",1.293901,1.595173,-0.274707,0.057040,0.135513
population_ordered_logistic,{'maxiter': 582},1.349075,1.702819,-0.486956,0.020996,0.092835
linear_regression,{'alpha': 8.87672855432774},1.464940,1.730648,-0.517784,-0.003573,0.239878
ordered_logistic,{'alpha': 9.652153198008865},1.547336,1.826667,-0.718277,-0.014586,0.214148


  history


...

,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
lag1_fatigue,{},0.950893,1.424175,0.104593,0.549449
expanding_mean,{},1.025298,1.336863,0.211017,0.422289
global_mode,{},1.156250,1.544479,-0.053072,0.000000
global_mean,{},1.406250,1.640721,-0.188402,0.000000


  base


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
catboost_ordinal,"{'iterations': 120, 'depth': 6, 'learning_rate...",1.130952,1.466897,0.050067,0.119330
catboost_regressor,"{'iterations': 135, 'depth': 5, 'learning_rate...",1.215774,1.530931,-0.034678,0.056789
ordinal_forest,"{'n_estimators': 231, 'max_depth': 14, 'min_sa...",1.224702,1.542551,-0.050444,0.054346
ordered_logistic,{'alpha': 9.652153198008865},1.309524,1.642987,-0.191686,-0.005966
ordinal_rf,"{'n_estimators': 107, 'max_depth': 5, 'min_sam...",1.342262,1.644797,-0.194314,0.062787
linear_regression,{'alpha': 8.87672855432774},1.345238,1.622021,-0.161467,-0.004347
population_ordered_logistic,{'maxiter': 582},1.455357,1.903162,-0.598988,-0.095675


  history


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
linear_regression_history,{'alpha': 0.053105729583361565},0.883929,1.205148,0.358828,0.547277
ordered_logistic_history,{'alpha': 1.8260541557754824},0.883929,1.245229,0.315470,0.550537
catboost_ordinal_history,"{'iterations': 256, 'depth': 4, 'learning_rate...",0.888393,1.214373,0.348974,0.551214
population_ordered_logistic_history,{'maxiter': 211},0.889881,1.307032,0.245835,0.555856
ordinal_forest_history,"{'n_estimators': 391, 'max_depth': 5, 'min_sam...",0.891369,1.225352,0.337149,0.538609
ordinal_rf_history,"{'n_estimators': 322, 'max_depth': 3, 'min_sam...",0.891369,1.215598,0.347660,0.532419
catboost_regressor_history,"{'iterations': 361, 'depth': 4, 'learning_rate...",0.910714,1.242837,0.318098,0.507826


In [21]:
# --- Base vs history ablation (test MAE only) ---
# delta_mae = history - base; negative means history features improved test MAE.

history_ablation_summary = build_history_ablation_summary(
    ordinal_test_summary, ORDINAL_MODELS
)
if history_ablation_summary.empty:
    print('No paired base/history models found — run both §3 blocks first.')
else:
    print(
        'Base vs history paired comparison '
        '(delta_mae = history - base; negative = history helps)'
    )
    display(history_ablation_summary)


Base vs history paired comparison (delta_mae = history - base; negative = history helps)


,test_mae_base,test_mae_history,delta_mae
model,,,
population_ordered_logistic,1.455357,0.889881,-0.565476
linear_regression,1.345238,0.883929,-0.461310
ordinal_rf,1.342262,0.891369,-0.450893
ordered_logistic,1.309524,0.883929,-0.425595
ordinal_forest,1.224702,0.891369,-0.333333
catboost_regressor,1.215774,0.910714,-0.305060
catboost_ordinal,1.130952,0.888393,-0.242560
